# 🚢 Titanic Dataset — Exploratory Data Analysis (EDA)
**Dataset:** Titanic passenger survival data  
**Goal:** Understand the data, find patterns, and decide which features matter before modeling.

> EDA is the most important step in any ML pipeline. We never jump to modeling without first understanding our data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load Titanic dataset directly from URL (no Kaggle login needed)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()


## 1. Basic Overview
Let's understand the structure — how many rows, columns, and what data types we have.

In [ ]:
print("Columns:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nBasic Stats:\n")
df.describe()


## 2. Missing Values
Missing data is a core ML challenge. We must identify and handle it before training.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Percentage', ascending=False)
print(missing_df)


In [ ]:
# Visualize missing values
plt.figure(figsize=(8, 4))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap', fontsize=14)
plt.tight_layout()
plt.show()


**Observations:**
- `Age` has ~20% missing values → we'll fill with median later
- `Cabin` has ~77% missing → too sparse, we'll drop it
- `Embarked` has 2 missing → fill with mode


## 3. Target Variable Distribution
How many people survived vs died?

In [ ]:
plt.figure(figsize=(6, 4))
df['Survived'].value_counts().plot(kind='bar', color=['#E74C3C','#2ECC71'], edgecolor='black')
plt.xticks([0, 1], ['Did Not Survive', 'Survived'], rotation=0)
plt.title('Survival Count', fontsize=14)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(df['Survived'].value_counts())
print(f"\nSurvival Rate: {df['Survived'].mean()*100:.1f}%")


## 4. Survival by Gender
Women and children first — let's verify this in data.

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(x='Sex', y='Survived', data=df, palette=['#3498DB','#E91E63'])
plt.title('Survival Rate by Gender', fontsize=14)
plt.ylabel('Survival Rate')
plt.tight_layout()
plt.show()

print(df.groupby('Sex')['Survived'].mean())


**Key Finding:** Females had a ~74% survival rate vs ~19% for males. Gender will be a very important feature in our model.


## 5. Survival by Passenger Class

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(x='Pclass', y='Survived', data=df, palette='Blues_d')
plt.title('Survival Rate by Passenger Class', fontsize=14)
plt.ylabel('Survival Rate')
plt.xlabel('Passenger Class (1=First, 3=Third)')
plt.tight_layout()
plt.show()

print(df.groupby('Pclass')['Survived'].mean())


## 6. Age Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall age distribution
axes[0].hist(df['Age'].dropna(), bins=30, color='#3498DB', edgecolor='black', alpha=0.7)
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# Age by survival
df[df['Survived']==0]['Age'].dropna().hist(ax=axes[1], bins=30, alpha=0.6, color='#E74C3C', label='Died')
df[df['Survived']==1]['Age'].dropna().hist(ax=axes[1], bins=30, alpha=0.6, color='#2ECC71', label='Survived')
axes[1].set_title('Age by Survival')
axes[1].set_xlabel('Age')
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Correlation Heatmap

In [ ]:
# Encode categorical for correlation
df_corr = df.copy()
df_corr['Sex'] = df_corr['Sex'].map({'male': 0, 'female': 1})
df_corr['Embarked'] = df_corr['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

plt.figure(figsize=(9, 6))
sns.heatmap(df_corr[['Survived','Pclass','Sex','Age','SibSp','Parch','Fare']].corr(),
            annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()


## Summary of Key Findings

| Feature | Impact on Survival |
|---|---|
| Sex | Females survived at 74% vs males at 19% |
| Pclass | 1st class: 63%, 3rd class: 24% |
| Age | Children had higher survival rates |
| Fare | Higher fare → higher class → higher survival |

> **Next Step:** Use these insights in the modeling notebook (`logistic_regression_titanic.ipynb`)
